# Stage 4: KTO Training - Binary Feedback Alignment

**Cipher Code Kraken Training Pipeline - Stage 4 of 4 (Final Stage)**

KTO (Kahneman-Tversky Optimization) uses binary thumbs-up/down feedback to do final alignment. Unlike SimPO (which needs paired chosen/rejected), KTO works with unpaired binary signals -- simpler to collect and still effective.

**Input:** `cipher-grpo-merged` (from Stage 3)
**Output:** `cipher-final-merged` (input for GGUF export)

**Key features:**
- Binary feedback alignment (thumbs up/down)
- Label balance monitoring (~60/40 positive/negative is ideal)
- Comprehensive multi-prompt eval to detect catastrophic forgetting
- Final model quality validation across 5 creative code categories

**Runtime:** Colab Pro+ A100 40GB | ~1-2 hours | ~20 compute units

## Cell 1: Environment Setup
Install Unsloth and W&B. Log in to W&B for experiment tracking.

In [ ]:
!pip install unsloth
!pip install wandb
import wandb
wandb.login()

## Cell 2: Mount Drive and Copy Data
Copy GRPO-merged model, KTO dataset, and scripts from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy GRPO-merged model (output from Stage 3)
!cp -r /content/drive/MyDrive/cipher-models/cipher-grpo-merged ./

# Copy KTO dataset
!mkdir -p data/prompts
!cp /content/drive/MyDrive/cipher-data/kto_prompts.jsonl ./data/prompts/

# Copy scripts (slop_detector for eval)
!cp -r /content/drive/MyDrive/cipher-scripts/scripts ./

# Copy configs
!mkdir -p configs
!cp /content/drive/MyDrive/cipher-scripts/configs/kto_config.py ./configs/
!cp /content/drive/MyDrive/cipher-scripts/configs/__init__.py ./configs/

print("Data mounted and copied successfully")
!ls -la cipher-grpo-merged/ | head -5
!wc -l data/prompts/kto_prompts.jsonl

## Cell 3: Load GRPO-Merged Model
Load the model from Stage 3 output with QLoRA 4-bit quantization.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys
sys.path.append('/content')
from configs.kto_config import *

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
)
print(f"GRPO-merged model loaded. VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 4: Apply LoRA Adapter
Apply a fresh LoRA adapter for KTO training. Same rank/alpha as all previous stages.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=RANDOM_STATE,
)
model.print_trainable_parameters()
print(f"Post-LoRA VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 5: Load and Validate KTO Dataset
KTO format: `{"prompt": "...", "completion": "...", "label": true/false}`

Check label balance -- KTO works with imbalanced data but ~60/40 positive/negative is ideal. If heavily imbalanced (>80/20), KTO may undertrain on the minority class.

In [ ]:
from datasets import load_dataset

# KTO format: {"prompt": "...", "completion": "...", "label": true/false}
kto_dataset = load_dataset("json", data_files=KTO_DATA_PATH)
print(f"KTO examples: {len(kto_dataset['train'])}")

# Check label balance
labels = [ex['label'] for ex in kto_dataset['train']]
pos = sum(1 for l in labels if l)
neg = sum(1 for l in labels if not l)
total = len(labels)
print(f"\nLabel Distribution:")
print(f"  Positive (thumbs up):   {pos} ({pos/total*100:.0f}%)")
print(f"  Negative (thumbs down): {neg} ({neg/total*100:.0f}%)")

# KTO works with imbalanced data, but ~60/40 split is ideal
if pos / total > 0.8:
    print("\nWARNING: Dataset is heavily positive-skewed (>80%).")
    print("KTO may undertrain on negative examples. Consider adding more rejected samples.")
elif neg / total > 0.8:
    print("\nWARNING: Dataset is heavily negative-skewed (>80%).")
    print("KTO may undertrain on positive examples.")
else:
    print(f"\nLabel balance looks good for KTO training.")

# Show samples
print(f"\nSample positive: {kto_dataset['train'][0]['prompt'][:100]}...")
for ex in kto_dataset['train']:
    if not ex['label']:
        print(f"Sample negative: {ex['prompt'][:100]}...")
        break

## Cell 6: Configure and Run KTO Training
Run KTO with binary feedback data. This is the lightest training stage:
- `learning_rate=5e-6`: lowest LR in the pipeline (200x lower than SFT)
- `num_train_epochs=1`: single pass through the data
- `max_prompt_length=512`: truncate long prompts to save memory

In [ ]:
from trl import KTOTrainer, KTOConfig
import os

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

kto_config = KTOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=BF16,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    report_to=REPORT_TO,
    run_name=WANDB_RUN_NAME,
)

trainer = KTOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=kto_dataset["train"],
    args=kto_config,
)

print(f"Pre-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
trainer.train()
print(f"Post-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
print("KTO training complete!")

## Cell 7: Merge Final Model
Merge the KTO adapter into the base. This is the **final merged model** that goes to GGUF export.

**Complete pipeline chain:** base -> SFT-merged -> SimPO-merged -> GRPO-merged -> **final-merged**

In [ ]:
model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer)
print(f"Final merged model saved to {MERGED_OUTPUT_DIR}")

# Save to Drive
!cp -r {MERGED_OUTPUT_DIR} /content/drive/MyDrive/cipher-models/
print("Final merged model copied to Drive")
print(f"\nNext step: Run notebook 06_gguf_export.ipynb to convert to GGUF for Ollama")

## Cell 8: Comprehensive Multi-Prompt Evaluation
Test across 5 different prompt types to check for catastrophic forgetting (Pitfall 2).

The model should produce creative, high-quality code for ALL prompt types -- not just the ones it was trained on. If any prompt type produces slop (high slop_score), the model may have overfit.

**Prompt categories tested:**
1. Three.js particle system + GSAP easing
2. Three.js 3D text + ScrollTrigger
3. WebGL shader + GSAP camera
4. Lenis smooth scroll + parallax
5. Creative 404 page + Three.js background

In [ ]:
# Test across multiple prompt types to check for catastrophic forgetting
FastLanguageModel.for_inference(model)

try:
    from scripts.slop_detector import slop_score
    has_slop_detector = True
except ImportError:
    has_slop_detector = False
    print("WARNING: slop_detector not available. Showing response lengths only.")

test_prompts = [
    "Create a Three.js particle system that responds to mouse movement with GSAP-powered easing transitions",
    "Build a hero section with 3D text that rotates on scroll using Three.js and GSAP ScrollTrigger",
    "Create a WebGL shader that generates a noise-based terrain with GSAP-controlled camera dolly on scroll",
    "Implement a Lenis smooth scroll page with GSAP ScrollTrigger pinned sections and parallax images",
    "Build a creative 404 page with a Three.js animated background and GSAP text animations",
]

results = []
for prompt in test_prompts:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Prompt: {prompt[:60]}...")
    print(f"Response length: {len(response)} chars")
    
    if has_slop_detector:
        score = slop_score(response)
        print(f"Slop score: {score['score']:.1f} (is_slop: {score['is_slop']})")
        creative_signals = [s for s in score['signals'] if s.startswith('-')]
        slop_signals = [s for s in score['signals'] if s.startswith('+')]
        print(f"Creative signals: {len(creative_signals)}")
        print(f"Slop signals: {len(slop_signals)}")
        results.append(score)
    
    print(f"Preview: {response[:200]}...")
    print("---")

if has_slop_detector and results:
    avg_score = sum(r['score'] for r in results) / len(results)
    slop_count = sum(1 for r in results if r['is_slop'])
    print(f"\n{'='*60}")
    print(f"FINAL EVAL SUMMARY")
    print(f"Average slop score: {avg_score:.1f}")
    print(f"Slop detections: {slop_count}/{len(results)}")
    if slop_count > 0:
        print("WARNING: Some prompts triggered slop detection. Review outputs carefully.")
    else:
        print("All prompts passed slop detection. Model quality looks good!")
    print(f"{'='*60}")